# Wish Generation LLM Evaluation

This notebook evaluates different LLM models for personalized wish generation.

## Metrics Tracked
- **latency_ms**: Time per prompt
- **tokens_in / tokens_out**: Inference cost basis
- **quality_score**: LLM-as-a-judge focusing on:
  - Warmth and personalization
  - Age-appropriateness
  - Safety (no hallucinated dangerous/offensive content)
- **monthly_estimate**: Cost for 2B requests

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "prototype" / "src"))

from src.clients.llm_client import LLMClient
from src.evaluation import (
    calculate_monthly_cost_estimate,
    evaluate_quality_with_judge,
    log_llm_evaluation,
    setup_mlflow,
)
from src.prompts import (
    generate_wish_prompt,
    generate_wish_quality_judge_prompt,
)
from src.test_samples import get_test_profiles, get_test_recommendations_for_wish
from src.utils import load_env_from_repo_root

# Load .env file from repository root
load_env_from_repo_root()

In [ ]:
setup_mlflow("wish_generation_eval")

In [ ]:
test_profiles = get_test_profiles()
test_recommendations = get_test_recommendations_for_wish()

In [ ]:
models_to_evaluate = []

if os.getenv("OPENAI_API_KEY"):
    models_to_evaluate.append({
        "name": "openai-gpt-3.5-turbo",
        "client": LLMClient(
            base_url="https://api.openai.com/v1",
            api_key=os.getenv("OPENAI_API_KEY"),
            model="gpt-3.5-turbo"
        ),
        "cost_per_1m_tokens_in": 1.5,
        "cost_per_1m_tokens_out": 2.0,
    })
    models_to_evaluate.append({
        "name": "openai-gpt-4-turbo",
        "client": LLMClient(
            base_url="https://api.openai.com/v1",
            api_key=os.getenv("OPENAI_API_KEY"),
            model="gpt-4-turbo-preview"
        ),
        "cost_per_1m_tokens_in": 10.0,
        "cost_per_1m_tokens_out": 30.0,
    })

if os.getenv("TOKEN_FACTORY_API_KEY") and os.getenv("TOKEN_FACTORY_BASE_URL"):
    models_to_evaluate.append({
        "name": "tokenfactory-llama-3-8b",
        "client": LLMClient(
            base_url=os.getenv("TOKEN_FACTORY_BASE_URL"),
            api_key=os.getenv("TOKEN_FACTORY_API_KEY"),
            model="meta-llama/Meta-Llama-3-8B-Instruct"
        ),
        "cost_per_1m_tokens_in": 0.1,
        "cost_per_1m_tokens_out": 0.1,
    })

vllm_base_url = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
if vllm_base_url:
    models_to_evaluate.append({
        "name": "vllm-llama-2-7b",
        "client": LLMClient(
            base_url=vllm_base_url,
            api_key=None,
            model=os.getenv("VLLM_MODEL_7B", "meta-llama/Llama-2-7b-chat-hf")
        ),
        "cost_per_1m_tokens_in": 0.0,
        "cost_per_1m_tokens_out": 0.0,
    })
    models_to_evaluate.append({
        "name": "vllm-llama-2-13b",
        "client": LLMClient(
            base_url=vllm_base_url,
            api_key=None,
            model=os.getenv("VLLM_MODEL_13B", "meta-llama/Llama-2-13b-chat-hf")
        ),
        "cost_per_1m_tokens_in": 0.0,
        "cost_per_1m_tokens_out": 0.0,
    })

In [ ]:
judge_client = None
if os.getenv("OPENAI_API_KEY"):
    judge_client = LLMClient(
        base_url="https://api.openai.com/v1",
        api_key=os.getenv("OPENAI_API_KEY"),
        model="gpt-3.5-turbo"
    )
elif os.getenv("TOKEN_FACTORY_API_KEY") and os.getenv("TOKEN_FACTORY_BASE_URL"):
    judge_client = LLMClient(
        base_url=os.getenv("TOKEN_FACTORY_BASE_URL"),
        api_key=os.getenv("TOKEN_FACTORY_API_KEY"),
        model="meta-llama/Meta-Llama-3-8B-Instruct"
    )

In [ ]:
results = []

for model_config in models_to_evaluate:
    model_name = model_config["name"]
    client = model_config["client"]

    for kid_profile, gift_recommendation in zip(test_profiles, test_recommendations):
        prompt = generate_wish_prompt(kid_profile, gift_recommendation)

        try:
            response, metrics = client.generate(prompt, temperature=0.9, max_tokens=200)

            quality_score = None
            if judge_client:
                judge_prompt = generate_wish_quality_judge_prompt(kid_profile, response)
                quality_score = evaluate_quality_with_judge(judge_client, judge_prompt)

            run_id = log_llm_evaluation(
                model_name=model_name,
                prompt=prompt,
                response=response,
                metrics=metrics,
                quality_score=quality_score,
                tags={
                    "kid_name": kid_profile.name,
                    "kid_age": str(kid_profile.age),
                    "task": "wish_generation",
                }
            )

            monthly_cost = None
            if metrics.get("tokens_in") and metrics.get("tokens_out"):
                avg_tokens_in = metrics["tokens_in"]
                avg_tokens_out = metrics["tokens_out"]
                monthly_cost = calculate_monthly_cost_estimate(
                    tokens_in_per_request=avg_tokens_in,
                    tokens_out_per_request=avg_tokens_out,
                    cost_per_1m_tokens_in=model_config["cost_per_1m_tokens_in"],
                    cost_per_1m_tokens_out=model_config["cost_per_1m_tokens_out"],
                )

            results.append({
                "model": model_name,
                "kid_name": kid_profile.name,
                "kid_age": kid_profile.age,
                "latency_ms": metrics.get("latency_ms", 0),
                "tokens_in": metrics.get("tokens_in"),
                "tokens_out": metrics.get("tokens_out"),
                "quality_score": quality_score,
                "monthly_cost_usd": monthly_cost,
                "run_id": run_id,
            })

        except Exception as e:
            results.append({
                "model": model_name,
                "kid_name": kid_profile.name,
                "error": str(e),
            })

In [ ]:
import pandas as pd

df_results = pd.DataFrame(results)
print("Evaluation Results Summary:")
print(df_results.groupby("model").agg({
    "latency_ms": "mean",
    "quality_score": "mean",
    "monthly_cost_usd": "mean",
}).round(2))